### Human lymph node - cell2location study: B cells, mesenteric LN, ISG hotspot (mp_lymphoid02)

Rewrite of **notebooks_ccc_lymphoid_liana/ccc02_study.ipynb** with the class libraries. It continues **mp_lymphoid01_prepare.ipynb**: the cell2location map (**sp.h5ad**), the models and the region labels computed there are read, not recomputed.

#### Libs: **src/libs**

- **single_cell_lib** - Class **SingleCell**: generic Scanpy, Squidpy, cell2location and LIANA methods; **DEFAULT_PARAMS**
- **single_cell_lymphoid_lib** - Class **SingleCellLymphoid(SingleCell)**: lymph node / lymphoid-organ methods; **LYMPHOID_PARAMS**, **LYMPHOID_LINEAGE**

| section | content |
|---|---|
| 1 | read the previous calculations (cell2location map + mp_lymphoid01 regions) and the scRNA reference |
| 2 | mLN = mesenteric lymph node: naive / memory / GC genes by age group |
| 3 | B-cell study: signatures that separate the 10 B-cell subsets; marker dot plot |
| 4 | larger gene panel on the Visium section |
| 5 | reference statistics; panel by age group (dot plot lineage x age group); panel UMAP by tissue x age |
| 6 | AICDA (AID) by B subset, tissue, donor and chemistry |
| 7 | gene filter, NB regression (reference signatures), posterior, training plot, QC |
| 8 | cell2location mapping, posterior, QC |
| 9 | cell abundance on the slide |
| 10 | interferon-stimulated-gene (ISG) hotspot |
| 11 | Figure 1 (UMAP + dot plot), Figure 2 (spatial maps), Figure 3 (gene vs abundance correlation) |
| 12 | dot plot lineage x NMF factors |

Figures and tables of this notebook have the prefix **mp_lymphoid02_**.

#### Cell2location continue

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BayraktarLab/cell2location/blob/master/docs/notebooks/cell2location_tutorial.ipynb)


This study is part of the Cell-Cell Communication (CCC) project and continues the first tutorial, ccc01_tutorial.ipynb (here: mp_lymphoid01_prepare.ipynb). It starts from the Visium object and the cell2location models saved there.

## Loading packages

In [1]:
import os, sys
from pathlib import Path
from importlib.metadata import version
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import squidpy as sq

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42
%config InlineBackend.print_figure_kwargs = {'bbox_inches': 'tight', 'dpi': 90}   # inline display only; files are saved at 300 dpi

ROOT0 = Path('/home/flavio/uv/cell_cell_communication')
ROOT_SRC = ROOT0 / 'src'
if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

from libs.single_cell_lymphoid_lib import SingleCellLymphoid, LYMPHOID_PARAMS, LYMPHOID_LINEAGE
from libs.matplotlib_figure_style import apply_figure_style
from libs.Basic import create_dir, pdwritecsv, pdreadcsv

pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 250)
print({p: version(p) for p in ['scanpy', 'squidpy', 'anndata', 'cell2location', 'scvi-tools', 'torch']})

## Parameters

**params** overrides **LYMPHOID_PARAMS** / **DEFAULT_PARAMS** (B-cell marker tiers: **marker_**, **unique_**, **shared_**; ISG hotspot: **isg_**).

In [2]:
project = 'lymphoid_tissue'

# ---- input files
fname_ref      = 'lymphoid_organ_sc.h5ad'                # root_data
c2l_dir        = 'lymph_nodes_analysis'                  # root_result / c2l_dir
ref_subdir     = 'reference_signatures'
map_subdir     = 'cell2location_map'
fname_map      = 'sp.h5ad'                               # previous calculation: cell2location map (mp_lymphoid01 / ccc01)
tsv_regions_prev = 'mp_lymphoid01_region_labels.tsv'              # regions of mp_lymphoid01 (root_result)
visium_id      = 'V1_Human_Lymph_Node'

# ---- output files
prefix = 'mp_lymphoid02_'
fig_bcell_dotplot  = prefix + 'bcell_subset_markers_dotplot.png'
fig_genes_visium   = prefix + 'visium_genes_{k}.png'
fig_mln_dotplot    = prefix + 'mln_panel_by_celltype_age.png'
fig_panel_umap     = prefix + 'genes_umap_by_tissue_age.png'
fig_ref_history    = prefix + 'ref_training_history.png'
fig_ref_qc         = prefix + 'ref_qc.png'
fig_map_history    = prefix + 'map_training_history.png'
fig_map_qc         = prefix + 'map_qc.png'
fig_abundance      = prefix + 'celltype_abundance_8types.png'
fig_slide_3types   = prefix + 'spatial_Tnaive_Bnaive_FDC.png'
fig_isg            = prefix + 'ifn_hotspot_summary.png'
fig_umap_dotplot   = prefix + 'celltype_umap_dotplot.png'
fig_spatial_maps   = prefix + 'celltype_spatial_maps.png'
fig_rho            = prefix + 'gene_celltype_spatial_correlation.png'
fig_factor_dotplot = prefix + 'nmf_factors_lineage_dotplot_n{n}.png'
csv_bcell_markers  = prefix + 'bcell_subset_unique_markers.csv'
tsv_mln_panel      = prefix + 'mln_panel_by_celltype_age.tsv'
csv_aicda_subset   = prefix + 'aicda_by_subset_tissue.csv'
csv_aicda_gc       = prefix + 'aicda_gc_by_donor.csv'
tsv_isg            = prefix + 'ifn_hotspot_celltype_enrichment.tsv'
tsv_rho            = prefix + 'gene_celltype_spatial_correlation.tsv'
tsv_factor_load    = prefix + 'nmf_factor_loadings_n{n}.tsv'

params = dict(
    accelerator='auto',
    clip_quantile=99.2,
    # B-cell subset markers (within-B comparison)
    marker_min_gene_frac=0.01, marker_padj=0.05, marker_min_pct_in=0.30,
    unique_max_pct_other=0.25, unique_min_log2fc=1.0,
    shared_max_pct_other=0.50, shared_min_log2fc=0.58,
    donor_min_cells=20,
    # ISG hotspot
    isg_quantile=0.97, isg_link_factor=1.6, isg_fdr=0.05,
    coloc_n_facts=(12,),
)

apply_figure_style()
scl = SingleCellLymphoid(ROOT0, project, params=params)
root_data, root_figure, root_result = scl.root_data, scl.root_figure, scl.root_result
results_folder = root_result / c2l_dir
ref_run_name = results_folder / ref_subdir
run_name = results_folder / map_subdir
print(scl)
print((run_name / fname_map).exists(), run_name / fname_map)

### 1. Read the previous calculations

The Visium object is the one saved by ccc01 / mp_lymphoid01 (results/lymph_nodes_analysis/cell2location_map/sp.h5ad), with cell2location abundances and per-cell-type layers. The raw data are lymph node data generated by 10X and presented in [Kleshchevnikov et al (section 4, Fig 4)](https://www.biorxiv.org/content/10.1101/2020.11.15.378125v1). The raw dataset can be conveniently downloaded and imported using scanpy. See [this tutorial](https://cell2location.readthedocs.io/en/latest/notebooks/cell2location_short_demo.html) for a more extensive and practical example of data loading (multiple visium samples).

The map is read **without** its 34 per-cell-type expected-expression layers (dense, about 5.6 GB; not used here): **scl.read_h5ad_slots**. The region labels are those of mp_lymphoid01.

In [3]:
adata_vis = scl.read_h5ad_slots(fname_map, root=run_name, skip=('layers',), verbose=True)
scl.abundance_to_obs(adata_vis)                       # q05 abundance per cell type -> obs

df_regions = pdreadcsv(tsv_regions_prev, root_result).set_index('barcode')
adata_vis.obs['region_cluster'] = df_regions.loc[adata_vis.obs_names, 'region_cluster'].astype(str).astype('category')
adata_vis.obs['region_name'] = df_regions.loc[adata_vis.obs_names, 'region_name'].astype('category')
scl.summary(adata_vis)

In [4]:
print(adata_vis.var.shape[0], "genes;", adata_vis.uns['spatial'].keys())
adata_vis.var.head(3)

In [5]:
adata_vis.obs[['region_cluster', 'region_name']].value_counts().sort_index().to_frame('n_spots')

### B-cell gene

Pan B-cell gene and protein markers used to identify B lymphocytes include CD19, MS4A1 (which encodes CD20), CD79A, CD79B, PAX5, and CD22. [1](https://www.biocompare.com/Editorial-Articles/570578-A-Guide-to-B-Cell-Markers/), [2](https://www.panglaodb.se/markers.html?cell_type='B%20cells'), [3](https://www.rpubs.com/daniloceschin/Markers_BCells)

Key B-Cell Gene and Surface Markers

* Core Pan B-Cell Markers:
  * CD19: Expressed from the early pro-B cell stage through mature B cells; acts as a coreceptor for B-cell receptor (BCR) signaling.
  * MS4A1 (CD20): Encodes the CD20 calcium channel protein; present from the pre-B stage to mature B cells and a primary target for immunotherapies (e.g., rituximab).
  * CD79A / CD79B: Encodes Igα and Igβ, essential signaling components associated with the surface B-cell receptor complex.
  * PAX5: A master transcription factor critical for B-cell commitment and identity.
  * CD22: A sialic acid-binding lectin expressed on mature and developing B cells. [1](https://apps.allenimmunology.org/aifi/resources/imm-health-atlas/cell-type-descriptions/b-cells-and-plasma-cells/), [2](https://www.antibodies.com/primary-antibodies/cell-markers/immune-cell-markers/b-cell-markers), [3](https://www.abcam.com/en-us/technical-resources/research-areas/marker-guides/b-cell-markers), [4](https://www.youtube.com/watch?v=JUq3K8rtcso&vl=en&t=178), [5](https://www.origene.com/research-areas/b-cell-lineage), [6](https://www.rpubs.com/daniloceschin/Markers_BCells), [7](https://www.biocompare.com/Editorial-Articles/570578-A-Guide-to-B-Cell-Markers/)
* Stage-Specific and Subset Markers:
  * Naïve B Cells: IGHD (IgD), IGHM (IgM), CD27−, IL4R, FCER2 (CD23).
  * Memory B Cells: CD27+, CD24+, and switched or unswitched immunoglobulins (IgG, IgA, IgE).
  * Plasma Cells / Plasmablasts: Downregulate CD19 and MS4A1/CD20 while upregulating CD38, SDC1 (CD138), PRDM1 (Blimp-1), and XBP1. [1](https://www.antibodies.com/primary-antibodies/cell-markers/immune-cell-markers/b-cell-markers), [2](https://www.biocompare.com/Editorial-Articles/570578-A-Guide-to-B-Cell-Markers/), [3](https://apps.allenimmunology.org/aifi/resources/imm-health-atlas/cell-type-descriptions/b-cells-and-plasma-cells/)

### See also

notebook: ccc00b_immune_cells.ipynb

### The reference: lymphoid_organ_sc.h5ad


**lymphoid_organ_sc.h5ad** is the single-cell reference for the cell2location paper's **lymph node / tonsil analysis**. 

It holds **73,260 cells × 10,237 genes**, 1.66 GB on disk, and contains both raw counts and a fitted reference model.

#### Expression matrix (**X**)

- **Raw UMI counts.** The matrix is sparse (CSR, float32) and every value is a whole number. In the first 2,000 cells the counts run from 1 to 1,276, with about 1,080 detected genes per cell.

- **Genes are already filtered.** 

  - Gene symbols are the index and Ensembl IDs are in **var['GeneID-2']**. 

  - The **n_counts** and **n_genes** columns were computed on the full gene set before filtering, so they are slightly higher than the row sums of **X** (e.g. 3,148 vs 3,122).

- **Cell QC:** 
  - median 3,483 UMIs and 1,657 genes per cell; 

  - minimum 581 UMIs and 698 genes.
  - Mitochondrial fraction (**percent_mito**) is 0–15%, median 4%. 
  - Every cell has **predicted_doublet = False**, so doublets were already removed.

#### Studies and tissues (**obs**)

| Study | Tissue | 3′ (3GEX) | 5′ (5GEX) |
|---|---|---|---|
| King_2020 | Tonsil | 2,002 | 27,703 |
| James_2020 | Mesenteric LN | 10,250 | 4,184 |
| Park_2020 | Spleen / iLN / mLN / tLN | 15,506 | 13,615 |

- **23 samples from 13 donors.** Donor A16 alone contributes 29,121 cells.

- **Ages** run from 0–4 years to 65–70 years.
- **Sex:** 48,545 female and 24,715 male cells.
- **Extra columns:** B-cell receptor fields (**BCELL_CLONE**, **ISOTYPE**, **IGH_MU_FREQ**, **VDJsum**), cell-cycle scores and phase, five Leiden clusterings, and several generations of annotation columns.


1. King HW, Orban N, Riches JC, Clear AJ, Warnes G, Teichmann SA, James LK. Single-cell analysis of human B cell maturation predicts how antibody class switching shapes selection dynamics. Sci Immunol. 2021 Feb 12;6(56):eabe6291. doi: 10.1126/sciimmunol.abe6291. PMID: 33579751.

2. James KR, Gomes T, Elmentaite R, Kumar N, Gulliver EL, King HW, Stares MD, Bareham BR, Ferdinand JR, Petrova VN, Polański K, Forster SC, Jarvis LB, Suchanek O, Howlett S, James LK, Jones JL, Meyer KB, Clatworthy MR, Saeb-Parsy K, Lawley TD, Teichmann SA. Distinct microbial and immune niches of the human colon. Nat Immunol. 2020 Mar;21(3):343-353. doi: 10.1038/s41590-020-0602-z. Epub 2020 Feb 17. PMID: 32066951; PMCID: PMC7212050.

Gastrointestinal microbiota and immune cells interact closely and display regional specificity, but little is known about how these communities differ with location. Here, we simultaneously assess microbiota and single immune cells across the healthy, adult human colon, with paired characterization of immune cells in the mesenteric lymph nodes, to delineate colonic immune niches at steady-state. We describe distinct T helper cell activation and migration profiles along the colon and characterize the transcriptional adaptation trajectory of T regulatory cells between lymphoid tissue and colon. Finally, we show increasing B cell accumulation, clonal expansion and mutational frequency from cecum to sigmoid colon, and link this to the increasing number of reactive bacterial species.

3. Park JE, Botting RA, Domínguez Conde C, Popescu DM, Lavaert M, Kunz DJ, Goh I, Stephenson E, Ragazzini R, Tuck E, Wilbrey-Clark A, Roberts K, Kedlian VR, Ferdinand JR, He X, Webb S, Maunder D, Vandamme N, Mahbubani KT, Polanski K, Mamanova L, Bolt L, Crossland D, de Rita F, Fuller A, Filby A, Reynolds G, Dixon D, Saeb-Parsy K, Lisgo S, Henderson D, Vento-Tormo R, Bayraktar OA, Barker RA, Meyer KB, Saeys Y, Bonfanti P, Behjati S, Clatworthy MR, Taghon T, Haniffa M, Teichmann SA. A cell atlas of human thymic development defines T cell repertoire formation. Science. 2020 Feb 21;367(6480):eaay3224. doi: 10.1126/science.aay3224. PMID: 32079746; PMCID: PMC7611066.

The thymus provides a nurturing environment for the differentiation and selection of T cells, a process orchestrated by their interaction with multiple thymic cell types. We used single-cell RNA sequencing to create a cell census of the human thymus across the life span and to reconstruct T cell differentiation trajectories and T cell receptor (TCR) recombination kinetics. Using this approach, we identified and located in situ CD8αα+ T cell populations, thymic fibroblast subtypes, and activated dendritic cell states. In addition, we reveal a bias in TCR recombination and selection, which is attributed to genomic position and the kinetics of lineage commitment. Taken together, our data provide a comprehensive atlas of the human thymus across the life span with new insights into human T cell development.


#### Cell-type annotations

- **Subset** (34 types) is the label set the regression model was trained on. The largest groups are 
  - B_mem (13,476), 
  - B_naive (8,924),
  - T_CD4+_naive (6,012),
  - B_Cycling (4,765) and
  - T_CD4+_TfH (4,690). 
  - It also covers
    - germinal-centre B cells (DZ, LZ, prePB, preGC), 
    - T-cell types (TfH, Tfr, Treg, CD8 types), 
    - NK/NKT/ILC, 
    - myeloid cells (DCs, macrophages, monocytes, mast cells) and 
    - stromal cells (Endo, FDC, VSMC). 
    - Rare types include ILC/Mast (18), VSMC (40) and DC_CCR7+ (42).

- **Subset_Broad** (38 types) is finer, despite its name. 
  - For example, it splits the endothelium into ACKR1+, CXCL12+ and lymphatic cells.

- **ILC / Mast label swap in Subset_Broad:**
  - Cell by cell, the 617 **Subset** ILC cells are exactly the 617 **Subset_Broad** Mast cells, and the 18 **Subset** Mast cells are the 18 **Subset_Broad** ILC cells.
  - Markers show that **Subset** is correct: its ILC cells express *IL7R* (94% of cells) and *KLRB1* (92%) and no *TPSAB1* (0%), while its 18 Mast cells are all *TPSAB1*+ (100%; *KIT* 78%).
  - So **Subset_Broad** has these two labels swapped; **Subset** (the labels the regression model is trained on) is fine.

#### Fitted reference model

This file already contains the trained cell2location reference model, so you don't need to retrain it.

- **uns['regression_mod']** is a **RegressionNBV4Torch** model with 57 factors: 
  - 34 cell types plus 
  - 23 sample effects.

- Its posterior means are:
  - **gene_factors** (57 × 10,237)
  - **sample_scaling** (23 × 1)
  - **tech_scaling** (2 × 10,237, one row per chemistry, 3′ and 5′)
  - **gene_E** (1 × 10,237)

- **In var:**
  - 34 **mean_cov_effect_Subset_*** columns hold each cell type's expression signature. 
  - These are the columns to pass to cell2location as the reference. 
  - There are also 23 **mean_sample_effectSample_*** columns, plus **n_cells** and **nonz_mean**.

#### Embeddings and metadata

- **Embeddings:**
  - **obsm['X_pca']** (100 components), 
  - **obsm['X_umap']**, and 
  - a neighbour graph built with 69 neighbours (**obsp** connectivities and distances).
- **Colour palettes** are stored in **uns** for the main categories.
- **Unusable columns:** **GeneName-2** holds numbers instead of gene names, and the **feature_types** / **gene_ids-*** columns are mostly NaN. Use the var index for gene names instead.

#### To extract the signatures for spatial mapping:

```python
sig = adata.var[[c for c in adata.var if c.startswith('mean_cov_effect_Subset_')]]
sig.columns = sig.columns.str.replace('mean_cov_effect_Subset_', '')
```


In [6]:
adata_ref = scl.read_h5ad(fname_ref, set_self=False, verbose=True)
scl.reindex_reference_var(adata_ref)                  # symbols -> var['symbol'], Ensembl IDs -> index
scl.add_tissue_age_group(adata_ref)                   # obs['age_group']
adata_ref

In [7]:
cols = ['barcode', 'batch', 'Donor', 'ID', 'n_counts', 'n_genes', 'new_celltype', 'percent_mito',
        'CellType', 'Tissue', 'VDJsum', 'Method', 'Population', 'PrelimCellType', 'PrelimCellType_new',
        'Age', 'age_group', 'Sex', 'Study', 'Subset', 'Subset_Broad', 'Subset_all', 'Subset_int', 'Subset_print',
        'S_score', 'Sample', 'cell_cycle_diff', 'doublet_score', 'predicted_doublet']
adata_ref.obs[cols].head(5).T

### 2. mLN = Mesenteric LN

The mesenteric Lymph Nodes are located in the abdomen, draining the intestines

#### mLN only: naive / memory / GC markers by age group (adult, old adult, elder)

- genes are looked up by symbol

**Caveat:** the means below are over all mLN cells (T, B and myeloid together), so differences between age groups mostly reflect cell-type composition; *LEF1*, *TCF7*, *CCR7* and *CD28* are largely T-cell genes here. 'old adult' is a single donor (302c). Normalisation here is `sc.pp.normalize_total` (row sums after gene filtering), slightly different from `panel_lognorm` (`obs['n_counts']`).


#### **Why mesenteric LN?**

mLN is the only tissue in the reference that covers more than one age group: adult, old adult (1 donor) and elder (4 donors). 

Tonsil has only children, and spleen, iLN and tLN have only the adult donor A16. 

So mLN is the only place in the reference where age-related changes can even be looked at.

**What the genes measure?** 

Together they track the classic signs of an ageing immune system: fewer naive cells, more differentiated or memory cells, and changes in germinal-centre activity.

| Gene | What it marks | Expected change with age |
|---|---|---|
| **LEF1**, **TCF7** | transcription factors of naive and stem-like memory T cells (TCF7 also drives TfH) | decrease, as naive T cells decline after the thymus shrinks with age |
| **CCR7**, **SELL** (CD62L) | lymph-node homing receptors of naive and central-memory T cells (and naive B cells) | decrease with fewer naive cells, or stay stable in central memory |
| **CD27** | memory B cells; naive and early-memory T cells | B cells: increase with more memory; T cells: decrease as late effector cells (CD27−) accumulate |
| **CD28** | main co-stimulatory receptor of T cells | decrease; CD28-negative T cells accumulate with age and chronic antigen exposure |
| **AICDA** | enzyme that drives antibody mutation and class switching in germinal centres | lower germinal-centre activity in older donors, although mLN keeps ongoing gut-driven germinal centres |

**Limits on interpreting it:**

- **Mixed cell types.** The cell compares averages over all mLN cells, so B and T cells are pooled. CD27 and CCR7 mean different things in B and T cells, and a shift in cell-type composition alone can change the averages. Comparing within cell types (`Subset`, e.g. T_CD4+_naive, B_mem) would be more informative.

- **Few donors.** "old adult" is one donor (1,825 cells), and adult and elder come from different studies (Park vs James). Differences can reflect donor or study as much as age.
- **Chemistry.** The reference mixes 3′ and 5′ libraries. That cell normalises to log1p(CP10k), which doesn't correct for the chemistry difference.



In [8]:
mln_genes = ["LEF1", "CCR7", "SELL", "TCF7", "CD27", "CD28", "AICDA"]
df_mln_mean, df_mln_frac, df_mln = scl.calc_df_tissue_age_expr(mln_genes, tissue='mLN', adata=adata_ref)
mln_order = list(df_mln_mean.index)
print(">> df_mln: mLN cells x genes (log1p CP10k)", df_mln.shape, mln_order)

df_mln.groupby("age_group")[mln_genes].mean().reindex(mln_order).round(3)

In [9]:
# fraction of mLN cells with expression > 0
df_mln_frac

In [10]:
df_mln.head(3)

## 3. B-cell study

### Which genes are unique when comparing the B cell types? post-study

GC DZ, GC LZ, GC prePB, B cycling, preGC, naive, activated, B IFN, B mem, and B plasma

**scl.prepare_subset_markers**: the 10 B subsets (38,309 cells), symbols as var_names, X = log1p(CP10k), genes detected in >= 1% of B cells, one-vs-rest Wilcoxon (**rank_genes_groups**, pts=True) within the B cells. **calc_df_group_pct_mean**: % expressing and mean CP10k per subset.

In [11]:
bcell_types = ["B_GC_DZ", "B_GC_LZ", "B_GC_prePB", "B_Cycling", "B_preGC", "B_naive", "B_activated", "B_IFN", "B_mem", "B_plasma"]

adata_bcell = scl.prepare_subset_markers(adata_ref, bcell_types)
print(adata_bcell.shape, adata_bcell.obs.Subset.value_counts().reindex(bcell_types).to_dict())
print("genes tested:", adata_bcell.n_vars)

df_pct, df_mean = scl.calc_df_group_pct_mean(adata_bcell, bcell_types)
print(df_pct.shape)
df_pct.head(6)

### Signatures: genes that are unique to one B subset

**calc_df_subset_markers** (thresholds in **params**): for every subset, % in the subset, max % in the other subsets, the next-highest subset, log2FC vs that subset (mean CP10k, pseudocount 0.05), Wilcoxon log2FC vs the rest of B and adjusted p; tiers **unique** / **enriched (shared)**; for the candidates, the donors in which the gene is still highest in that subset (**donors_top**).

In [12]:
df_all, dff = scl.calc_df_subset_markers(adata_bcell, df_pct, df_mean, bcell_types, fname=csv_bcell_markers)

df_tier_summary = dff.groupby(["group", "tier"], observed=True).size().unstack(fill_value=0)
df_tier_summary

In [13]:
# B cells per donor and subset (donor consistency uses pairs with >= donor_min_cells cells)
df_all.attrs['df_donor_counts']

In [14]:
scl.print_subset_markers(dff, bcell_types)

In [15]:
# summary table: unique genes per subset (first 10 rows of every group)
cols = ["group", "gene", "pct_in", "max_pct_other", "log2fc_vs_next", "donors_top"]

dftab = dff[dff.tier == "unique"].groupby("group", observed=True)[cols]
dftab.head(10).round(3)

### Subsets without unique genes: best genes vs the next subset, and naive vs memory head to head

In [16]:
for g in ["B_naive", "B_mem", "B_GC_LZ", "B_activated"]:
    df_b = scl.calc_df_best_vs_next(df_all, g)
    print(g, "| best vs next:", ", ".join(f"{r.gene}({r.pct_in*100:.0f}/{r.max_pct_other*100:.0f}%,{r.log2fc_vs_next:.2f} vs {r.next_mean_type})" for r in df_b.itertuples()))
    df_w = scl.wtab(g, adata_bcell.uns["rank_genes_groups"]).head(8)
    print("   top Wilcoxon vs rest-B:", ", ".join(f"{i}({row.lfc:.1f})" for i, row in df_w.iterrows()), "\n")

df_nm = scl.calc_df_rank_genes(adata_bcell, groupby="Subset", groups=["B_naive"], reference="B_mem")
print("naive>mem:", df_nm.head(10)[["names", "logfoldchanges"]].round(2).values.tolist())
print("mem>naive:", df_nm.sort_values("scores").head(10)[["names", "logfoldchanges"]].round(2).values.tolist())

### Dot plot of the markers (4 per subset)

In [17]:
markers = {
    "B_GC_DZ":     ["SUGCT", "MME", "SH3TC1", "USP53"],
    "B_GC_LZ":     ["LMO2", "RGS13", "GMDS", "PLEK"],
    "B_GC_prePB":  ["FRZB", "BAIAP2L1", "ARHGAP10", "RASSF6"],
    "B_Cycling":   ["CDK1", "TOP2A", "RRM2", "UBE2C"],
    "B_preGC":     ["CCND2", "MIR155HG", "CRIP3", "LILRA4"],
    "B_naive":     ["TCL1A", "IGHD", "FCER2", "IGHM"],
    "B_activated": ["CD69", "CD83", "NR4A1", "NFKBIZ"],
    "B_IFN":       ["IFIT3", "IFIT1", "IFI44L", "MX2"],
    "B_mem":       ["CD27", "TNFRSF13B", "GPR183", "CLECL1"],
    "B_plasma":    ["DERL3", "FKBP11", "IGHG2", "PRDX4"]}

dp = scl.dotplot_bcell_markers(adata_bcell, markers, fname=fig_bcell_dotplot)
plt.show()

### Comparing the 10 B-cell subsets against each other only

Not against T or myeloid cells, so a gene counts as "unique" only if it separates one B subset from all the other B subsets. Per subset, three criteria are combined:

1. a one-vs-rest Wilcoxon test within the B-cell compartment (log1p CP10k);
2. a specificity filter against the *next-highest* B subset (thresholds below);
3. a donor-consistency check, because cell-level tests overstate significance when cells come from few donors.

The test ran on 38,309 B cells and 8,375 genes (genes detected in ≥ 1% of B cells). Subset sizes range from 74 (`B_GC_prePB`) to 13,476 (`B_mem`).

A gene counts as **unique** if it meets all of these:
- adjusted p < 0.05;
- detected in ≥ 30% of the subset's cells;
- detected in ≤ 25% of cells in *every* other B subset;
- at least 2× the mean CP10k of the next-highest B subset (log2FC ≥ 1).

**Enriched (shared)** relaxes the last two criteria to ≤ 50% and ≥ 1.5×.

"Donors" is the number of donors (each with ≥ 20 cells of that subset) in which the gene is still highest in that subset (`scl.donor_consistency`).

Six of the 10 B-cell subsets have clearly unique genes: plasma, cycling, IFN, pre-plasmablast (prePB), pre-GC and dark zone (DZ). Naive B, memory B and GC light zone (LZ) have **no** gene that is unique against all the other B subsets, and activated B has only one marginal one (*MYADM*). These four are defined by combinations of genes, or by contrast with one neighbouring subset. Naive markers such as *TCL1A*, *IGHD* and *FCER2* stand out only when naive cells are compared directly with memory cells, because activated, pre-GC and IFN B subsets also carry them.



| Subset (cells) | Unique genes (top) | % in subset / max % in other B subsets | Donors |
|---|---|---|---|
| **B_plasma** (1,094) | *DERL3, FKBP11, IGHG2, IGHGP, PRDX4, TXNDC5, FNDC3B* | *DERL3* 92 / 18 | 6/6 |
| **B_Cycling** (4,765) | *CDK1, TOP2A, RRM2, UBE2C, CCNA2, TYMS, ASF1B* (132 genes in total) | *CDK1* 56 / 4 | 8/8 |
| **B_IFN** (199) | *IFIT3, IFIT1, IFI44L, IFI6, MX2, STAT1, XAF1* | *IFIT3* 56 / 2 | 2/2 |
| **B_GC_prePB** (74) | *FRZB, BAIAP2L1, ARHGAP10, BTNL9, CNR1* | *BAIAP2L1* 39 / 7 | 1/1 |
| **B_preGC** (404) | *CCND2, MIR155HG, CRIP3, SLAMF1, LILRA4* | *CCND2* 37 / 13 | 5/5 |
| **B_GC_DZ** (2,500) | *SH3TC1, USP53, SLC2A5* (unique); *SUGCT, MME* (CD10), shared with cycling | *SH3TC1* 36 / 16 | 8/8 |
| **B_activated** (3,575) | *MYADM* only; otherwise immediate-early genes *CD69, CD83, FOS, NR4A1, NFKBIZ* | *MYADM* 37 / 21 | 7/9 |
| **B_GC_LZ** (3,298) | none; best are *LMO2* (51 / 27%, 2.8× DZ), *RGS13, GMDS, PLEK* (shared with DZ, prePB, cycling) | — | — |
| **B_mem** (13,476) | none; best are *TNFRSF13B* (TACI), *CLECL1, GPR183* (shared with IFN B) | — | — |
| **B_naive** (8,924) | none; *IGHD, FCER2, TCL1A, IGHM* are shared with the activated, pre-GC and IFN B cells | — | — |


**Naive vs memory, head to head:** *TCL1A* (log2FC 5.5), *IGHD* (3.2), *FCER2* (2.5) and *CD72* (1.7) are higher in naive cells. *TNFRSF13B* (3.4), *CD27* (3.3) and *GPR183* (2.4) are higher in memory cells. So the practical rule for naive B is IGHD⁺ FCER2⁺ TCL1A⁺ CD27⁻.

**Caveats:**
- **Cycling and IFN describe a cell state, not a lineage.** Their unique genes are the cell-cycle and interferon programs. DZ cells are proliferating too, which is why *SUGCT* and *MME* are shared with `B_Cycling` and not unique to DZ.
- **Some subsets are too small to check across donors.** `B_GC_prePB` has 74 cells, and only one donor has ≥ 20 of them; `B_IFN` has 199 cells, and only two donors qualify. Their markers are consistent only within those few donors.
- ***LILRA4* in pre-GC B is suspicious.** It is a classic plasmacytoid-DC marker, so it may come from contaminating cells or doublets, not genuine B-cell expression.
- **The p-values are optimistic.** The test treats each cell as independent, although cells come from only 13 donors. Rely on the effect-size filters and the donor counts more than on the p-values.
- ***TCL1A* and *CD27* aren't specific to naive or memory cells.** By mean CP10k, *TCL1A* is highest in GC DZ and cycling cells, and *CD27* is highest in GC DZ, prePB and plasma cells. Each is specific only in the naive-vs-memory contrast.
- **Dot-plot genes (4 per subset)** were picked from the unique list where one existed, otherwise from the best shared or head-to-head genes.

Outputs:
- `figures/mp_lymphoid02_bcell_subset_markers_dotplot.png`: the dot plot above.
- `results/mp_lymphoid02_bcell_subset_unique_markers.csv`: 502 candidate markers in two tiers, 219 "unique" and 283 "enriched (shared)". For each gene: % in the subset, max % in other subsets, the next-highest subset, log2FC vs that subset, adjusted p and the donor count.


In [18]:
del adata_bcell, df_all
import gc; gc.collect();

### 4. A larger gene panel on the Visium section

CD20 --> MS4A1
CD21 --> CR2
CD23 --> FCER2
HIST1H3B (current HGNC symbol: H3C2). Both datasets use the older symbol **HIST1H3B**; H3C2 is in neither. HIST1H3B is in the reference but not in the Visium genes (filtered out), so it appears only in the reference plots.

In [19]:
# Memory B Cells: CD27+, CD24+,
# Core Pan B-Cell: CD19, MS4A1 (CD20):, CD79A / CD79B, PAX5, CD22
# Naive B Cells: IGHD (IgD), IGHM (IgM), CD27-, IL4R, FCER2 (CD23)
genes_plot = ["AICDA", "CCR7",
              "CD19", "MS4A1", "CR2", "CD22", "FCER2", "CD24", "CD27", "CD28",
              "CD38", "CD52", "CD53", "CD79A", "CD79B", "CDS2",
              "IGHD", "IGHM", "IL4R", "LEF1", "SELL", "PAX5", "TCF7",
              # extra: B-subset marker genes (table above), to observe in the cell-type signatures
              "DERL3", "CDK1", "IFIT3", "IFIT1", "BAIAP2L1", "ARHGAP10", "FRZB", "CCND2", "SH3TC1", "USP53", "SLC2A5", "MYADM",
              "TNFRSF13B", "CLECL1", "GPR183", "TCL1A", "RRM2", "HIST1H3B", "LILRA4", "CRIP3",
              "IGHG2", "IGHGP"]
genes_plot = list(np.unique(genes_plot))

found_vis, _ = scl.resolve_gene_ids(genes_plot, adata_vis)
print("genes", len(genes_plot), "| in the Visium map", len(found_vis), "| missing:", sorted(set(genes_plot) - set(found_vis)))

### Visium spatial plot per gene (8 per figure)

Colour = counts clipped at the 99.2th percentile per gene (non-zero percentile for very sparse genes).

In [20]:
genes_shown, figs = scl.spatial_scatter_genes(found_vis, adata_vis, ncols=4, nframe=8, fname=fig_genes_visium)
plt.show()

### 5. Reference statistics

In [21]:
obs_cols = ['Study', 'Tissue', 'Method', 'Donor', 'Sample', 'Age', 'Sex', 'Subset_Broad', 'Subset', 'phase',
            'predicted_doublet']
for c, vc in scl.value_counts(obs_cols, adata_ref).items():
    print(f"\n## {c} ({adata_ref.obs[c].nunique(dropna=False)} levels)")
    print(vc.to_string())

In [22]:
cols = ['n_counts', 'n_genes', 'percent_mito', 'doublet_score']
adata_ref.obs[cols].describe().round(2)

In [23]:
print("X:", scl.x_info(adata_ref))
scl.calc_df_sample_stats(adata_ref)

### Naive / memory / GC genes by age group

All cells (fraction > 0 and mean log1p(CP10k) per age group), then mLN by cell type (lineage) x age group.

In [24]:
df_age_mean = scl.calc_df_gene_age_tissue_groups(mln_genes, adata_ref)
df_age_mean.pivot_table(index='gene', columns=['tissue', 'age_group'], values='mean_log1p_cp10k', sort=False)

In [25]:
lineage = LYMPHOID_LINEAGE
df_mln_long, df_mln_plot, df_mln_age = scl.calc_df_panel_celltype_age(mln_genes, tissue='mLN', adata=adata_ref,
                                                                       fname=tsv_mln_panel)
fig = scl.plot_dotplot_celltype_age(df_mln_plot, lineage=lineage, df_age=df_mln_age, fname=fig_mln_dotplot)
plt.show()

### Split by age group and tissue: the larger panel on the reference UMAP

The Visium slide **V1_Human_Lymph_Node** is one lymph node section from one donor, so its spots carry no tissue or age labels. Tissue and age exist only in **lymphoid_organ_sc.h5ad**. The gene panel is therefore shown on the reference UMAP, one column per tissue × age group, with the same 99.2-percentile clipping used for the Visium plots.

**Age groups** (`scl.age_group()`, using the lower bound of each Age bin):

| Group | Age bins | Cells |
|---|---|---|
| babies | 0–4y | 16,237 |
| young | 5–9y | 13,468 |
| adult | 20–25y | 29,121 |
| old adult | 40–44y | 1,825 |
| elder | 55–60y, 60–64y, 65–70y | 12,609 |

**Tissue and age are almost entirely confounded.** Only 8 of the 25 tissue × age combinations contain cells:

- Tonsil exists only as babies (2 donors) and young (5 donors), all from King_2020.
- Spleen, iLN and tLN are adult only, and all come from the single donor A16 (Park_2020).
- Only mLN spans three age groups: adult (A16), old adult (one donor, 302c, 1,825 cells) and elder (four donors).

So a difference between columns is just as likely to come from tissue, study (King, Park or James) or donor as from age.

**Plot choices:**

- **Protein names are replaced by gene symbols:** CD20 → *MS4A1*, CD21 → *CR2*, CD23 → *FCER2*; the histone gene is *HIST1H3B* (H3C2). `resolve_gene_ids` / `clip_for_plot` warn about, and drop, any name not found in `var['symbol']`.
- **Sparse genes.** When more than 99.2% of cells have zero counts for a gene, the 99.2-percentile cap is 0 and the row would be blank; `percentile_cap` then uses the 99.2th percentile of the non-zero values.
- **Expression is normalised.** The reference holds raw counts from mixed 3′ and 5′ chemistry, so colour = log1p(counts per 10k) per cell (library size = `obs['n_counts']`). Each gene has one colour scale shared across all columns, so columns can be compared directly.
- **Colour map.** `cmap_num=2` is blue-yellow-red, the same map as the Visium plots. `cmap_num=3` (library default) is a sequential light-grey → dark-red ramp; `cmap_num=1` (RdBu_r) is diverging and not recommended, because expression has no midpoint.

To split the slide itself by group, one would build separate cell2location reference signatures per group and map each onto the Visium spots. Given the confounding, the result would mostly reflect tissue and study, not age.

Outputs: **figures/bcell_genes_umap_by_tissue_age.png** (cell below) and the per tissue × age table `dfg` (n_cells, % expressing, mean log1p(CP10k)) returned in the Figure 1 cell.


### Tissue types

#### Secondary lymphoid organs in the reference

Primary lymphoid organs (bone marrow and thymus) are where lymphocytes develop; none is in this reference. Spleen, tonsil and lymph nodes are **secondary** lymphoid organs, where immune responses start.

- Spleen: the largest secondary lymphoid organ. It
  - filters blood,
  - removes old red blood cells, and
  - stores platelets and white blood cells; its white pulp holds B-cell follicles and T-cell zones.

- Tonsil: lymphoid tissue aggregates in the pharynx (throat)
  - that act as a first line of defence against inhaled or ingested pathogens; paediatric tonsils have many active germinal centres.

#### Lymph nodes (LNs)

- mLN: mesenteric lymph nodes (abdomen, draining the intestines)
- iLN: inguinal (or iliac) lymph nodes; verify the abbreviation against the Park_2020 metadata
- tLN: thoracic lymph nodes (chest, e.g. tracheobronchial / mediastinal). The A16 samples come from a healthy organ donor, so "tumour-draining" does not apply; verify against the source study.


In [26]:
found_ref, _ = scl.resolve_gene_ids(genes_plot, adata_ref)
print(len(found_ref), 'of', len(genes_plot), 'genes found in the reference; not found:', [g for g in genes_plot if g not in found_ref])

In [27]:
apply_figure_style(sizes=(8, 7, 6))
fig = scl.plot_panel_umap_by_tissue_age(genes_plot, adata_ref, fname=fig_panel_umap, cmap_num=2)
plt.show()

### 6. AID, the enzyme encoded by *AICDA*

**Observation (UMAP panel above):** *AICDA* is mostly in dark-zone (DZ) and cycling B cells, a little in pre-GC B cells, and only in tonsils.

AID is the single best marker of germinal-centre (GC) B cells, so the DZ / cycling part matches biology. "Only in tonsils", though, has to be checked against how this reference was assembled, because tissue is confounded with study, donor age and 10x chemistry. The code cell below quantifies:
- *AICDA* detection per tissue within each B subset, especially within GC B cells;
- how many GC B cells each tissue contributes at all;
- whether the signal holds per donor and per chemistry (3′ vs 5′);
- whether the *AICDA*⁺ cells labelled naive or memory are single-UMI noise or GC-like cells under a non-GC label.

**Summary:** *AICDA* in DZ and cycling B cells is confirmed. "Only in tonsils" is not: within GC B cells, the spleen and lymph nodes express *AICDA* at about the same rate. Tonsils dominate only because they contain almost all of the GC B cells in this reference.

#### What AICDA does

*AICDA* encodes **AID (activation-induced cytidine deaminase)**, an APOBEC-family enzyme. It converts cytosine to uracil in single-stranded DNA at actively transcribed immunoglobulin loci. That one reaction starts two processes:
- **Somatic hypermutation (SHM):** point mutations in the antibody variable regions. Proliferating DZ cells acquire the mutations, and light-zone (LZ) cells with higher affinity are selected. This drives affinity maturation.
- **Class-switch recombination (CSR):** double-strand breaks in switch regions change the antibody class from IgM/IgD to IgG, IgA or IgE. CSR can also happen outside germinal centres, in activated B cells at the T–B border.

Other points:
- **Regulation:** AID is induced by CD40L together with IL-4 or IL-21, and by TLR signals. It is tightly controlled after translation, through nuclear export and degradation.
- **Disease:** loss of AID causes hyper-IgM syndrome type 2 (no class switching, no hypermutation). Off-target AID activity causes oncogenic mutations and translocations such as *MYC–IGH* in Burkitt lymphoma and diffuse large B-cell lymphoma.

#### Critique

**1. "Only in tonsils" is mostly a cell-composition effect.** Half of tonsil B cells are GC B cells (DZ/LZ/prePB/Cycling), against 0.5–4.4% in the other tissues:

| Tissue | GC B cells | Share of B cells |
|---|---|---|
| Tonsil | 10,304 | 50% |
| tLN | 107 | 4.4% |
| Spleen | 122 | 2.9% |
| iLN | 69 | 2.0% |
| mLN | 35 | 0.5% |

That is why tonsils contribute 96% of all *AICDA*⁺ cells. Within GC B cells, per donor × chemistry library (≥ 20 GC cells each), detection is similar everywhere:
- tonsil, 7 donors: 21–47%;
- spleen, one donor (A16), 3′ / 5′ libraries: 46% / 42%;
- iLN (A16): 30% / 27%;
- tLN (A16): 27% / 19%.

So *AICDA* marks GC B cells in every tissue. Paediatric tonsils simply have many active germinal centres, consistent with constant antigen exposure in childhood.

**2. Tissue is confounded with donor, age and study.**
- All tonsils come from King 2020, from children.
- Spleen, iLN and tLN all come from one adult organ donor (A16, Park 2020).
- mLN comes from James 2020 and from A16 (Park 2020). It has only 35 GC B cells and no donor with ≥ 20, so it can't be evaluated.

Chemistry is not the explanation: within tonsil GC B cells, 3′ and 5′ data give 29.2% vs 29.3%.

**3. Detection is limited by sparse capture.** In the GC subsets, 52–79% of *AICDA*⁺ cells have a single UMI. So "41% of DZ cells" understates true expression, and a cell without *AICDA* reads may still express it. Ranking DZ above LZ (41% vs 9% in tonsil) is supported, and fits AID acting mainly during DZ proliferation. The exact percentages are partly set by how much RNA was captured.

**4. "A little in pre-GC" is not supported:** 0 of 404 `B_preGC` cells have any *AICDA* read.

**5. The *AICDA*⁺ naive and memory cells are probably not true naive or memory cells.**
- **"Naive":** 23 of 8,924 cells (0.26%). The 19 in tonsil carry a full GC program, unlike the *AICDA*⁻ tonsil naive cells:

| Gene | *AICDA*⁺ "naive" | *AICDA*⁻ naive |
|---|---|---|
| *RGS13* | 68% | 2% |
| *BCL6* | 63% | 9% |
| *GCSAM* | 68% | 6% |
| *MKI67* | 58% | 0% |

  They look like mislabelled GC or cycling cells.
- **"Memory":** 96 of 13,476 cells (0.7%), 78 of them in tonsil. The tonsil ones show a partial GC program:

| Gene | *AICDA*⁺ memory | *AICDA*⁻ memory |
|---|---|---|
| *RGS13* | 24% | 9% |
| *LMO2* | 29% | 8% |
| *MKI67* | 19% | 1% |

  Plausible explanations: cells that recently left the GC (memory precursors), mislabelled LZ cells, or ambient RNA in tonsil suspensions, which are rich in GC transcripts.
- Their doublet scores are not elevated, but doublets were already filtered from this reference, so that check carries little weight.

**6. mRNA is not activity.** AID is regulated mostly after translation, so *AICDA* transcripts don't show that hypermutation or class switching is happening. This reference includes BCR data that could test it directly: `IGH_MU_FREQ` (IGH mutation frequency) and `ISOTYPE`. That check has not been run yet.

**7. These are cell-level percentages from few donors.** For mLN and the A16 tissues, the GC B-cell counts are too small for firm conclusions.

The code cell below writes two tables:
- `results/mp_lymphoid02_aicda_by_subset_tissue.csv`: % *AICDA*⁺ and cell counts for each B subset × tissue;
- `results/mp_lymphoid02_aicda_gc_by_donor.csv`: % *AICDA*⁺ within GC B cells per donor, study and chemistry.

The naive / memory GC-program tables (point 5) were computed separately and are not reproduced by that cell.


In [28]:
aicda = scl.calc_gene_by_subset_tissue("AICDA", adata_ref, groups=bcell_types,
                                       fname_subset=csv_aicda_subset, fname_gc=csv_aicda_gc)
print(f"AICDA+ cells: {aicda['n_pos']:,}, of which in tonsil: {aicda['pct_pos_tonsil']}%")
print("tonsil GC B, % AICDA+ by chemistry:", aicda['chem_tonsil_gc'])
aicda['df_gc_share']

In [29]:
aicda['df_subset']

In [30]:
# % AICDA+ within GC B cells per donor x chemistry library with >= 20 GC B cells
aicda['df_gc_donor']

### 7. Gene filter and reference signatures

<div class="alert alert-info">
<b>Note!</b>
    
Before we estimate the reference cell type signature we recommend to perform very permissive genes selection. We prefer this to standard highly-variable-gene selection because our procedure keeps markers of rare genes while removing most of the uninformative genes.
</div>

The default parameters **cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12** are a good starting point, however, you can increase the cut-off to exclude more genes. 

To preserve marker genes of rare cell types we recommend low **cell_count_cutoff=5**, however, **cell_percentage_cutoff2** and **nonz_mean_cutoff** can be increased to select between 8k-16k genes.

In this 2D histogram, orange rectangle highlights genes excluded based on the combination of number of cells expressing that gene (Y-axis) and average RNA count for cells where the gene was detected (X-axis).

In this case, the downloaded dataset was already filtered using this method, hence no density under the orange rectangle (to be changed in the future version of the tutorial).

### cell2location: "Markers of rare genes" is a slip in the original; it should read "markers of rare **cell types**".

#### The problem it solves

The reference regression model estimates an expression signature for every cell type. The goal is to:

- **remove** genes with almost no signal, which add noise and computation.
- **keep** genes that identify any cell type, including rare ones.

Standard highly-variable-gene (HVG) selection doesn't serve the second goal well:

- **It keeps a fixed top-N** (e.g. 2,000 genes) ranked by dispersion across *all* cells. A gene expressed only in a population making up 0.5% of cells adds little to the global variance, so it can fall below the cutoff.
- **It is tuned for a different job.** HVG selection picks genes for clustering and embedding. Cell2location wants as many informative genes as possible, because more genes give better deconvolution of spots.
- **It runs on normalised, log-transformed data.** Cell2location models raw counts.

#### What `filter_genes` does

```python
from cell2location.utils.filtering import filter_genes
selected = filter_genes(adata_ref, cell_count_cutoff=5,
                        cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12)
```

It works on raw counts. For each gene it computes two numbers:

- `n_cells`: how many cells have a count above 0.
- `nonz_mean`: the mean count *in those expressing cells only*.

A gene is kept if **either** rule applies:

1. **Broadly detected:** it is expressed in more than 3% of all cells (`cell_percentage_cutoff2`). This keeps genes expressed in common cell types.
2. **Rare but strongly expressed:** it is expressed in more than 5 cells (`cell_count_cutoff`) **and** its mean in the expressing cells is above 1.12 (`nonz_mean_cutoff`). This rule is how rare-cell-type markers survive. A gene found in only 200 cells, but at several UMIs per cell, looks like a real marker. A gene found in 200 cells at exactly 1 UMI each looks like ambient RNA or dropout noise.

Genes that meet neither rule are discarded. These are genes detected in 5 or fewer cells, or detected in under 3% of cells at only about 1 count each.

The value 1.12 is chosen just above 1. A gene passes rule 2 only if at least some of its expressing cells have 2 or more UMIs, which separates real expression from scattered single-molecule background.

#### Reading the diagnostic plot

The function draws a 2D histogram of genes: log10(`nonz_mean`) on the x-axis and log10(`n_cells`) on the y-axis.

- A horizontal line marks the 3% threshold (`cell_percentage_cutoff2`); every gene above it is kept.
- Below that line, only genes to the right of the vertical 1.12 line (`nonz_mean_cutoff`) and above the 5-cell line (`cell_count_cutoff`) are kept: few cells, but high counts.
- The orange shading marks the excluded region (low detection, about 1 count per expressing cell).

#### In this lymphoid-organ reference

With 73,260 cells, 3% is about 2,198 cells. *AICDA* (3,332 cells) and *DERL3* (5,352 cells) pass by rule 1. *LILRA4* (pDC marker; 1,530 cells, mean 2.4 UMIs in expressing cells) passes only by rule 2, which is the kind of gene HVG selection might drop.

The filter (next cell) keeps all 10,237 genes, because the downloaded reference was already filtered this way (no density under the orange rectangle). So every gene of `selected_genes_lymphoid_organ.csv` that is present in the reference passes. Raising `nonz_mean_cutoff` or `cell_percentage_cutoff2` gives stricter selection.


In [31]:
n0 = adata_ref.n_vars
adata_ref, selected = scl.filter_genes_c2l(adata_ref, figsize=(5, 4))
plt.show()
print(f"genes {n0:,} -> {adata_ref.n_vars:,}")

### Estimation of reference cell type signatures (NB regression)

The signatures are estimated from scRNA-seq data, accounting for batch effect, using a Negative binomial regression model.

<div class="alert alert-block alert-message">
<b>Preparing anndata.</b>
    
First, prepare anndata object for the regression model.

This section repeats the ccc01 pipeline. With the models saved on disk it only reloads them (no training); the signatures are q50, as in ccc01.
</div>

In [32]:
mod_reg, adata_ref, ref_trained = scl.fit_or_load_reference(
    adata_ref, ref_run_name, labels_key='Subset', batch_key='Sample', categorical_covariate_keys=['Method'])
mod_reg.view_anndata_setup()

<div class="alert alert-block alert-message">
<b>Determine if the model needs more training.</b>
</div>

Here, we plot ELBO loss history during training, removing first 20 epochs from the plot.
This plot should have a decreasing trend and level off by the end of training. If it is still decreasing, increase `max_epochs`.

In [33]:
fig = scl.plot_c2l_history(mod_reg, 20, fname=fig_ref_history)
plt.show()

You can compute the 5%, 50% and 95% quantiles of the posterior distribution directly rather than using 1000 samples from the distribution (or any other quantiles). This speeds up application on large datasets and requires less memory - however, posterior mean and standard deviation cannot be computed this way.

In [34]:
df_inf_aver, summary_used = scl.get_reference_signatures(adata_ref, prefer=(scl.params['c2l_signature_summary'],))
print(f"Using {summary_used} signatures:", df_inf_aver.shape)
df_inf_aver.head(6).T.head(8)

<div class="alert alert-block alert-message">
<b>Examine QC plots.</b>
</div>

1. Reconstruction accuracy to assess if there are any issues with inference. This 2D histogram plot should have most observations along a noisy diagonal.

2. The estimated expression signatures are distinct from mean expression in each cluster because of batch effects. For scRNA-seq datasets which do not suffer from batch effect (this dataset does), cluster average expression can be used instead of estimating signatures with a model. When this plot is very different from a diagonal plot (e.g. very low values on Y-axis, density everywhere) it indicates problems with signature estimation.

In [35]:
figs = scl.plot_c2l_qc(mod_reg, summary_name="q50", figsize=(3, 3), fname=fig_ref_qc)

### 8. Cell2location: spatial mapping

<div class="alert alert-block alert-message">
<b>Find shared genes and prepare anndata.</b>
Subset both anndata and reference signatures:
</div>

<div class="alert alert-info">
<b> Important </b>
    
To use cell2location spatial mapping model, you need to specify 2 user-provided hyperparameters (**N_cells_per_location** and **detection_alpha**) 

For detailed guidance on setting these hyperparameters and their impact see [the flow diagram and the note](https://github.com/BayraktarLab/cell2location/blob/master/docs/images/Note_on_selecting_hyperparameters.pdf).
</div>

**Choosing hyperparameter `N_cells_per_location`**

It is useful to adapt the expected cell abundance `N_cells_per_location` to every tissue. This value can be estimated from paired histology images and as described in the note above. Change the value presented in this tutorial (`N_cells_per_location=30`) to the value observed in your tissue.

**Choosing hyperparameter `detection_alpha`**

To improve accuracy & sensitivity on datasets with large technical variability in RNA detection sensitivity within the slide/batch - you need to relax regularisation of per-location normalisation (use `detection_alpha=20`). High technical variability in RNA detection sensitivity is present in your sample when you observe the spatial distribution of total RNA count per location that doesn't match expected cell numbers based on histological examination.

We initially opted for high regularisation (`detection_alpha=200`) as a default because the mouse brain & human lymph node datasets used in our paper have low technical effects and using high regularisation strength improves consistency between total estimated cell abundance per location and the nuclei count quantified from histology ([Fig S8F in cell2location paper](https://static-content.springer.com/esm/art%3A10.1038%2Fs41587-021-01139-4/MediaObjects/41587_2021_1139_MOESM1_ESM.pdf)). However, in many collaborations, we see that Visium experiments on human tissues suffer from technical effects. This motivates the new default value of `detection_alpha=20` and the recommendation of testing both settings on your data (`detection_alpha=20` and `detection_alpha=200`).

In [36]:
region_obs = adata_vis.obs[['region_cluster', 'region_name']].copy()
mod_c2l, adata_vis, map_trained = scl.fit_or_load_mapping(adata_vis, df_inf_aver, run_name, read_layers=False)
adata_vis.obs[['region_cluster', 'region_name']] = region_obs.loc[adata_vis.obs_names]
mod_c2l.view_anndata_setup()

### ELBO loss history during training, first 1000 epochs removed

In [37]:
fig = scl.plot_c2l_history(mod_c2l, 1000, label='full data training', fname=fig_map_history)
plt.show()

<div class="alert alert-block alert-message">
<b>Assessing mapping quality.</b>
Examine reconstruction accuracy to assess if there are any issues with mapping.
The plot should be roughly diagonal, strong deviations will signal problems that need to be investigated.
</div>

In [38]:
figs = scl.plot_c2l_qc(mod_c2l, summary_name="q50", figsize=(3, 3), fname=fig_map_qc)

When integrating multiple spatial batches and when working with datasets that have substantial variation of detected RNA within slides (that cannot be explained by high cellular density in the histology), it is important to assess whether cell2location normalised those effects. You expect to see similar total cell abundance across batches but distinct RNA detection sensitivity (both estimated by cell2location). You expect total cell abundance to mirror high cellular density in the histology.

```python
fig = mod_c2l.plot_spatial_QC_across_batches()
```

### 9. Visualising cell abundance in spatial coordinates

<div class="alert alert-info">
Note
    
We use 5% quantile of the posterior distribution, representing the value of cell abundance that the model has high confidence in (aka 'at least this amount is present').
</div>

In [39]:
slide = scl.select_slide(adata_vis, visium_id)
cell_types_8 = ['B_Cycling', 'B_GC_LZ', 'T_CD4+_TfH_GC', 'FDC', 'B_naive', 'T_CD4+_naive', 'B_plasma', 'Endo']
fig = scl.plot_spatial_features(slide, cell_types_8, ncols=4, fname=fig_abundance)
plt.show()

In [40]:
# cell2location plotter: several cell types in one panel; C2L_COLORS = plot_spatial colour order
C2L_COLORS = ["#F0E442", "#D55E00", "#56B4E9", "#009E73", "#5A14A5", "#C8C8C8", "#323232"]
# FDC - a follicular dendritic cell is a non-bone marrow-derived, mesenchymal stromal cell, not a classical dendritic cell
clust_labels = ['T_CD4+_naive', 'B_naive', 'FDC']
fig = scl.plot_slide_loc_cell_types(slide, clust_labels, C2L_COLORS, fname=fig_slide_3types)
plt.show()

### 10. B_IFN: interferon-stimulated-gene (ISG) hotspot (IFIT1, IFIT3, ...)

**scl.run_isg_hotspot**: ISG score per spot = mean z-score of log1p(CP10k) of the ISGs found; hotspot = the largest spatially connected group of spots above the 97th percentile of the score (spots linked when closer than 1.6 x the spot spacing). Cell-type enrichment in the hotspot vs the rest of the slide: q05 abundance, Mann-Whitney U, BH FDR. Panels: ISG score, B_IFN abundance, *CXCL10*, DC_CCR7+ abundance (**params['isg_maps']**), and the log2FC bars (n.s. = FDR >= 0.05).

In [41]:
ISG_GENES = ["IFIT3", "IFIT1", "IFIT2", "IFI44L", "ISG15",
             "MX1", "MX2", "IFI6", "RSAD2", "OAS1", "STAT1", "XAF1"]

df_hot = scl.run_isg_hotspot(adata_vis, ISG_GENES, fname_fig=fig_isg, fname_tsv=tsv_isg)
plt.show()

In [42]:
df_hot.round(4).head(12)

### 11. Figures 1-3

Based on mean log1p(CP10k) and % of cells expressing per **Subset** in the scRNA reference (see the dot plot below). The extra B-subset marker genes in `genes_plot` follow the table in "Comparing the 10 B-cell subsets".

1. pan-B: *CD19*, *MS4A1* (CD20), *CD79A*, *CD79B*, *PAX5*, *CD22* (all B subsets, highest in GC B)
2. naive / follicular-mantle B: *IGHD*, *IGHM*, *FCER2* (CD23) (B_naive, B_activated, B_preGC)
3. germinal-centre B: *AICDA* (B_GC_DZ, B_Cycling), *IL4R* (B_GC_DZ), *CD38* (GC B and B_plasma)
4. FDC: *CR2* (CD21) (highest in FDC; lower in B_preGC, B_IFN, B_GC_LZ)
5. naive T: *CCR7*, *LEF1*, *SELL*, *TCF7* (T_CD4+_naive, T_CD8+_naive, T_Treg; *CCR7* also DC_CCR7+, *SELL* also DC_pDC, *TCF7* also T_CD4+_TfH_GC)
6. activated / GC T: *CD27*, *CD28* (T_TIM3+, T_CD4+_TfH_GC, T_TfR, T_Treg; *CD27* also B_GC_DZ)
7. pan-lymphocyte: *CD52*, *CD53* (broadly expressed, highest in B cells)
8. *CD24*: B_IFN, B_mem and FDC

*CDS2* doesn't mark any clear cell type (maximum mean 0.28 log1p(CP10k), in VSMC).


In [43]:
lineage = LYMPHOID_LINEAGE            # row order: cell types grouped by lineage
focus = ["B_GC_LZ", "B_GC_DZ", "B_Cycling", "B_preGC", "B_naive", "B_mem", "B_plasma",
         "T_CD4+_TfH_GC", "T_TIM3+", "FDC", "Macrophages_M1", "Monocytes"]
cmaps = scl.params['lineage_cmaps']

ct = adata_ref.obs["Subset"].astype(str).values
assert set(ct) == set(scl.lineage_order(lineage))
np.unique(ct)

#### Figure 1: reference UMAP by cell type + dot plot of the panel genes

In [44]:
dfg = scl.umap_cell_type_plot(adata_ref, genes_plot, lineage, focus, cmaps, fname=fig_umap_dotplot)
plt.show()
print(dfg.shape)
dfg.head(4)

#### Figure 2: spatial maps of the focus cell types (q05 abundance)

In [45]:
fig = scl.spatial_plot_bcell_types(adata_vis, focus, lineage, cmaps=cmaps, fname=fig_spatial_maps)
plt.show()

#### Figure 3: gene (Visium) vs cell-type abundance correlation

Spearman rho across spots between the log1p(CP10k) of each panel gene and the mapped q05 abundance. Caveat: cell2location inferred the abundances from these same counts, so part of each correlation is built in; read it as consistency, not as independent validation.

In [46]:
np.array(adata_vis.uns["mod"]["factor_names"])

In [47]:
df_rho = scl.spatial_plot_cell_abundance(adata_vis, adata_ref, genes_plot, focus, lineage, cmaps=cmaps,
                                         fname_fig=fig_rho, fname_tsv=tsv_rho)
plt.show()
df_rho.head(6)

### 12. Dot plot lineage x NMF factors

The NMF co-location of mp_lymphoid01 / ccc01 (12 factors, stored in **adata_vis.uns['mod_coloc_n_fact12']**). Relative weight of every cell type over the factors (rows sum to 1); rows grouped by lineage; factors ordered by the row of their top cell type and labelled with their top-2 cell types.

In [48]:
adata_vis, res_dict, coloc_edited = scl.run_colocation_cached(adata_vis, run_name)    # cached: no NMF training
for n in scl.params['coloc_n_facts']:
    fig, df_load = scl.plot_factor_dotplot(adata_vis, n, lineage=lineage, cmaps=cmaps,
                                           fname=fig_factor_dotplot.format(n=n), fname_tsv=tsv_factor_load.format(n=n))
    plt.show()
df_load.round(2)

In [49]:
# B-cell rows: in which compartment (factor) is each B subset?
df_b = df_load.loc[[c for c in bcell_types if c in df_load.index]]
pd.DataFrame({'top_factor': df_b.idxmax(axis=1), 'weight': df_b.max(axis=1).round(2)})